# 01 — Worker

**Run this unedited, every session.** It claims whatever the ledger says is
next and runs it.

There is nothing to configure. The cell, scene and seed come from the ledger,
which enforces stage order and prerequisites globally — which is precisely why
this is one notebook rather than eight.

If a session dies mid-run, do nothing: the row is left with a stale heartbeat
and the next session reclaims it. Interrupted runs **restart** rather than
resume, deliberately — the checkpoint omits the medium model, the codebooks and
the loop's schedule flags, so resuming would silently reinitialise β and
produce a run that looks complete and is a different experiment.


## 1. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/     the four scenes (original) + undistorted/  <- created below
#     dense/       M1 clouds, with SHA-256 sidecars
#     runs/        <cell>/<scene>/s<seed>/  -- one run, all of it together
#     analysis/    analyse.py output, figures, tables
#     run_ledger.json
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATASET_DIR  = f'{DRIVE_ROOT}/dataset'
DATA_UNDIST  = f'{DATASET_DIR}/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} not found. Check the folder name, or edit DRIVE_ROOT above.')
for d in (DATA_UNDIST, DENSE_DIR, f'{DRIVE_ROOT}/runs', ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)

# Export them so the `!` cells below resolve "$DRIVE_ROOT" as a real shell
# variable. Relying on IPython to substitute notebook variables into magics
# works until it doesn't, and when it doesn't it substitutes nothing and the
# command runs against a silently truncated path rather than failing.
os.environ.update(
    DRIVE_ROOT=DRIVE_ROOT, DATASET_DIR=DATASET_DIR, DATA_UNDIST=DATA_UNDIST,
    DENSE_DIR=DENSE_DIR, ANALYSIS_DIR=ANALYSIS_DIR, LOCAL_DATA=LOCAL_DATA,
    REPO_DIR=REPO_DIR, IMPL_DIR=IMPL_DIR,
)


def find_originals():
    """Locate the four scenes under dataset/, however they were arranged.

    Accepts the scenes directly under dataset/, or nested one level (e.g.
    dataset/SeathruNeRF_dataset/). Returns the directory that contains them.
    """
    candidates = [DATASET_DIR] + [
        os.path.join(DATASET_DIR, d) for d in sorted(os.listdir(DATASET_DIR))
        if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'undistorted'
    ]
    for base in candidates:
        if all(os.path.isdir(os.path.join(base, s)) for s in SCENES):
            return base
    return None


DATA_ORIG = find_originals()

# verify_undistort's T1 -- the check that would catch the undistortion gap --
# reads the *original* dataset. Point it at wherever it actually landed on
# Drive, or T1 reports "dataset not found" and the one check that matters here
# quietly stops testing anything.
if DATA_ORIG:
    os.environ['E3DGSUW_DATASET'] = DATA_ORIG

print('drive root :', DRIVE_ROOT)
print('originals  :', DATA_ORIG or 'NOT FOUND')
print('undistorted:', DATA_UNDIST)


## 2. GPU — must be an A100

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {name}  sm_{cap[0]}{cap[1]}')

# Every conclusion in this study is a between-cell contrast, and cells on
# different devices are not comparable. Stop now rather than produce a run
# that has to be discarded later.
assert 'A100' in name, f'Expected an A100, got {name!r}. Restart the runtime.'


## 3. Clone and build  *(a few minutes)*

In [ ]:
import os, subprocess

# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left
# sidebar) with a fine-grained read token, and toggle notebook access on.
# Read from Secrets rather than pasted into the cell: a pasted token is saved
# inside the .ipynb, which then travels wherever the notebook does.
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or None
    print('GITHUB_TOKEN: loaded from Colab Secrets')
except ImportError:
    pass                                  # not running under Colab
except Exception as e:                    # secret absent, or access not granted
    print(f'GITHUB_TOKEN: not available ({type(e).__name__}) -- '
          'fine for a public repo')

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

# Never let git fall back to an interactive credential prompt: in a notebook it
# hangs the cell indefinitely with nothing on screen to say why.
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}


def _redact(s):
    """Strip the token from git output -- git echoes the remote URL on failure,
    and notebook outputs are saved to the file and shared with it."""
    return s.replace(GITHUB_TOKEN, '***') if GITHUB_TOKEN else s


if os.path.isdir(REPO_DIR) and not os.path.isdir(f'{REPO_DIR}/.git'):
    raise RuntimeError(
        f'{REPO_DIR} exists but is not a git checkout -- probably a clone that '
        f'died partway. Delete it and re-run this cell.')

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git','clone','--depth','1',url,REPO_DIR],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'clone failed. If the repository is private, add a GITHUB_TOKEN '
            'secret in Colab and grant this notebook access.\n'
            f'{_redact(r.stderr)[-800:]}')
else:
    # Repoint the remote before pulling. The stored URL was written by an
    # earlier clone, which may have run without a token (or with a stale one);
    # injecting the token into `url` alone never reaches the pull.
    subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin',url],
                   check=True, env=env)
    r = subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'pull failed. If the repository is private, check the GITHUB_TOKEN '
            'secret is set and this notebook has access.\n'
            f'{_redact(r.stderr)[-800:]}')

# Fail here, naming the directory, rather than letting a later cell run from
# whatever the working directory happened to be.
assert os.path.isdir(IMPL_DIR), (
    f'clone produced no {IMPL_DIR}. Contents of {REPO_DIR}: '
    f'{sorted(os.listdir(REPO_DIR)) if os.path.isdir(REPO_DIR) else "missing"}')

os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())
print('cwd:', os.getcwd())


In [ ]:
# Builds diff_gaussian_rasterization_ms and simple_knn against whatever torch
# Colab ships -- deliberately NOT installing our own, which would risk a
# mismatch between torch's CUDA and the toolkit the extensions compile with.
# Takes a few minutes; must be repeated each session.
#
# chdir explicitly rather than via `%cd $IMPL_DIR`: a magic whose variable fails
# to expand reports the *current* directory and continues, so the build then
# runs from the wrong place and fails two steps later with a bare
# "tools/setup_colab.sh: No such file or directory".
import os
assert os.path.isdir(IMPL_DIR), (
    f'{IMPL_DIR} not found -- run the "Clone the repository" cell above first.')
os.chdir(IMPL_DIR)
print('building in', os.getcwd())
!bash tools/setup_colab.sh


In [ ]:
import importlib, torch
for m in ('diff_gaussian_rasterization_ms', 'simple_knn'):
    importlib.import_module(m)
print('extensions import OK  |  torch', torch.__version__,
      '| cuda', torch.version.cuda)


### 3b. The upstream checkout, for S0

`SS` is vanilla SeaSplat, and the worker runs **upstream's** trainer inside
**upstream's** tree. Both live in `/content`, which Colab wipes between
sessions, so this belongs here beside the CUDA build rather than in setup —
same lifetime, same reason.

Upstream imports `diff_gaussian_rasterization`; this repository uses
Mini-Splatting's fork as `diff_gaussian_rasterization_ms` (CD-13). The names
differ, so both install side by side and neither shadows the other, which is
what lets the reference run its own kernels in a shared session.

**One deviation, and it is a build fix rather than a change to the method.**
Recent libstdc++ releases dropped transitive `<cstdint>` includes that
upstream's CUDA sources relied on, so the tree does not compile on this
toolchain as cloned — the same defect this repository fixed in its own copies
(`docs/reproducibility_notes.md` §5c). `fix_upstream_includes` adds the missing
headers and prints every file it touched. An include of a header the
translation unit already depends on resolves no new declaration and changes no
behaviour; it changes whether the file compiles.

That is categorically different from `tools/instrument_reference.py`, which
adds printing to the densification loop and says in its own docstring that a
tree it has touched must not produce SS numbers. That one belongs to notebook
04 and lives at a different path.

A clone that is merely cloned looks correct right up until the worker claims
its first SS row, so the extension is imported here and the cell raises if it
is missing.


In [ ]:
import os, subprocess, importlib

VANILLA = '/content/seasplat_vanilla'
if not os.path.isdir(f'{VANILLA}/.git'):
    !rm -rf {VANILLA}
    !git clone -q --recursive https://github.com/dxyang/seasplat.git {VANILLA}
!cd {VANILLA} && git log -1 --format="SS reference at %H  %ad" --date=short

# Recent libstdc++ dropped transitive <cstdint>, which upstream's CUDA sources
# relied on. This repository fixed its own copies in tree; the checkout is
# cloned fresh each session and carries the defect. Additive includes only,
# and every file touched is printed -- the record of the deviation.
!python -m tools.fix_upstream_includes {VANILLA}

# Its own rasterizer, built here. Upstream imports
# `diff_gaussian_rasterization`; this repository uses Mini-Splatting's fork
# under the name `diff_gaussian_rasterization_ms` (CD-13). Different module
# names, so both install side by side and neither shadows the other -- which
# is what lets the reference run its own kernels while sharing the session.
#
# NOT -q. `pip install` without it swallows nvcc's diagnostics and leaves only
# "see the compiler output above", with no output above -- the exact reason
# this project spent a session chasing a build failure once already.
!pip install -v {VANILLA}/submodules/diff-gaussian-rasterization 2>&1 | tail -25

importlib.invalidate_caches()
try:
    import diff_gaussian_rasterization  # noqa: F401
    print('upstream rasterizer: importable')
except Exception as exc:
    raise SystemExit(f'upstream rasterizer did not build: {exc}')

print('NOT patched -- this is the tree S0 measures.')


## 4. Verify the rasterizer

In [ ]:
!python -m tools.verify_rasterizer


## 5. Stage the dataset locally

The loader reads every image at startup; from Drive that is markedly slower
than one bulk copy.


In [ ]:
import os, shutil, time
missing = [s for s in SCENES if not os.path.isdir(f'{DATA_UNDIST}/{s}')]
assert not missing, (
    f'Undistorted scenes missing: {missing}. Run 00_setup.ipynb first — the '
    f'scenes are OPENCV-model and will not load undistorted.')

os.makedirs(LOCAL_DATA, exist_ok=True)
t0 = time.time()
for s in SCENES:
    if not os.path.exists(f'{LOCAL_DATA}/{s}'):
        shutil.copytree(f'{DATA_UNDIST}/{s}', f'{LOCAL_DATA}/{s}')
print(f'staged in {time.time()-t0:.0f}s')
!python -m tools.run_ledger status --output_root "$DRIVE_ROOT"


## 6. Work

`--max_minutes` sits **below** the session limit so the loop stops claiming new
runs and exits cleanly rather than being killed mid-run. Raise it if your
sessions run longer.

Until the budget is set, every M2 cell is blocked and the queue says so — that
is expected during S1.

**S0 runs here too, and it does not use this codebase's trainer.** The first
rows the queue hands out are `SS` — vanilla SeaSplat. The worker runs the
**upstream** trainer inside the unpatched checkout `00_setup` §10 stages, then
measures the result with this campaign's harness, so their model is scored on
our metrics with one convention on both sides. It never runs
`train.py --cell SS`: that would train *our* implementation under A0's defaults
and file it as the reference control, after which `check_margin` would compare
A0 against A0 and certify the study's foundational claim from the code agreeing
with itself.

Each SS row is one vanilla training run, so S0 is about eleven hours. Nothing
else waits on it in the sense that matters — the ledger sequences it first
because A0's standing rests on it, and every later contrast is a difference
against A0.

**Two messages that look like failures and are not.**

`... blocked: SS is produced from the upstream checkout, and none is staged` —
run `00_setup` §10. `blocked` is recomputed on every claim, so cloning the
checkout releases those rows with no further bookkeeping; if it never appears,
the stage is skipped loudly and the queue starts at A0 rather than stalling.

`... is blocked: dense cloud missing` — run `00_setup` §9. Same mechanism.


In [ ]:
!python -m tools.run_queue \
    --output_root "$DRIVE_ROOT" \
    --data_root   "$LOCAL_DATA" \
    --max_minutes 200


## 7. After S1 — check the budget against what A0 actually built

**There is nothing to set here.** `n_bud` is fixed ahead of the campaign by the
binding rule — it must lie below the smallest count any other enabled mechanism
produces — and `run_ledger init` reads it from `configs/cells.json`. It is not
derived from A0.

This cell is a check, because the rule was applied to *estimated* cloud sizes.
A0's realised counts tell you how much room M2 actually has, and section 9 of
`00_setup` reports the clouds. If the budget no longer sits below them, lower
it and re-run the affected cells — the rule holds, the number follows the
measurement.


In [ ]:
import glob, csv, json, os, statistics
counts = []
for f in sorted(glob.glob(f'{DRIVE_ROOT}/runs/A0/*/s*/diagnostics.csv')):
    rows = list(csv.DictReader(open(f)))
    if rows:
        counts.append((f.split('/runs/')[1].rsplit('/', 1)[0],
                       int(rows[-1]['n_primitives'])))
for name, n in counts:
    print(f'{n:>12,}  {name}')
if counts:
    med = statistics.median(n for _, n in counts)
    led = json.load(open(f'{DRIVE_ROOT}/run_ledger.json'))
    bud = led.get('n_bud')
    print(f'\nA0 median {med:,.0f}')
    if bud:
        print(f'n_bud     {bud:,}  (from {led.get("n_bud_source") or "set-budget"})')
        print(f'M2 would remove {100 * (1 - bud / med):.1f}% of A0')
        for path in sorted(glob.glob(f'{DENSE_DIR}/*.json')):
            side = json.load(open(path))
            pts = side.get('kept')
            if pts:
                ok = 'binds' if bud < pts else 'DOES NOT BIND -- lower n_bud'
                print(f'  {os.path.basename(path)[:-5]:<26} cloud={pts:>9,}  {ok}')
    else:
        print('n_bud NOT SET -- configs/cells.json has no defaults.n_bud, so '
              'every m2 cell is blocked.')
else:
    print('No A0 diagnostics yet.')


---
Re-run this notebook in a fresh session to continue. Nothing changes between
sessions.
